# 补充评测 Notebook

**目标**：补跑缺失的评测数据，统一使用自定义 eval 脚本（不用 lm-eval）。

| Phase | 内容 | 预计耗时 | 状态 |
|-------|------|----------|------|
| **E8** | Targeted DPO 数据构建（从 SFT badcase 提取 DPO 对）| ~1 min | 待跑 |
| **E9** | Targeted DPO 训练（SFT 基座 + Badcase-Targeted DPO）| ~15 min | 待跑 |
| **E10** | Targeted DPO 评测（GSM8K + MATH-500，n=200）| ~20 min | 待跑 |
| **E5** | Group C Teacher DPO 数据补全（SFT 模型推理生成 rejected）| ~30 min | 待跑 |
| **E6** | Group C DPO 训练（SFT 基座 + Teacher DPO）| ~20 min | 待跑 |
| **E7** | Group C 评测（GSM8K + MATH-500，n=200）| ~20 min | 待跑 |
| **E4** | 汇总表 | ~1 min | — |
| **E1** | 1.5B Baseline（GSM8K + MATH-500，n=200）| ~15 min | ✅ 已完成 |
| **E2** | 7B Baseline via API（GSM8K + MATH-500，n=200）| ~20 min | ✅ 已完成 |
| **E3** | Group A DPO 评测（GSM8K + MATH-500，n=200）| ~20 min | ✅ 已完成 |

**评测方式**：`eval/gsm8k_eval.py` + `eval/math_eval.py`（自定义，与之前 n=200 的实验一致）。
**输出**：`logs/eval_supplement_*.json`

**Targeted DPO 设计说明**（E8-E10，优先跑）：
- 直接复用 SFT 模型在 GSM8K + MATH-500 评测中产生的 badcase（~550 条）
- chosen = 正确解法（gt_raw），rejected = 模型的错误推理（pred_raw）
- 无需额外推理，数据量较小（~500 对），max_steps=400 防过拟合
- 目标：验证「针对性纠错」是否比通用 DPO 更有效

**Group C 设计说明**（E5-E7）：
- Teacher chosen: Qwen3-235B-Thinking 对 GSM8K 训练集 1500 题的详细 CoT（已生成）
- Teacher rejected: 用 SFT 模型在同样 1500 题上推理，答错的作为 rejected
- 这样 rejected 反映 1.5B 模型的真实错误模式，DPO 信号更有效

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E0: 环境准备
# ═══════════════════════════════════════════════════════════════════
!pip install -q -U pip
!pip install -q "unsloth>=2025.1.0" "trl>=0.14.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" \
    "transformers>=4.49.0" "datasets>=3.2.0" "accelerate>=1.2.0" \
    "pyyaml>=6.0.2" "safetensors" "tqdm" "scipy" "sympy" "requests"

import torch, os, json, subprocess, sys
from pathlib import Path
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB')

# 全局常量
EVAL_N = '200'
BIT = ['--load_in_4bit']

def run_eval(cmd, label):
    """运行评测子进程，实时打印输出。"""
    cmd = [cmd[0], '-u'] + cmd[1:]
    print(f'  执行: {" ".join(cmd[-8:])}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'  ❌ {label} 失败 (exit {proc.returncode})')
        return False
    return True

def print_result(path, label):
    """打印评测结果摘要。"""
    if not os.path.isfile(path):
        print(f'  {label}: 结果文件不存在 ({path})')
        return
    d = json.load(open(path))
    acc = d.get('accuracy', d.get('macro_avg_accuracy', 'N/A'))
    total = d.get('total', 'N/A')
    if isinstance(acc, float):
        print(f'  {label}: {acc:.1%} ({total}题)')
    else:
        print(f'  {label}: {acc} (n={total})')

def is_eval_complete(output, expected=None):
    """检查评测输出文件是否存在且已完成（details 数量 >= expected）。"""
    if not os.path.isfile(output):
        return False
    try:
        d = json.load(open(output))
        details = d.get('details', [])
        if expected is not None and len(details) < int(expected):
            print(f'  ⏩ {output}: 已有 {len(details)}/{expected} 条，续跑...')
            return False
        return True
    except Exception:
        return False


In [ ]:
# ── 0.2 挂载 Drive + 同步代码 + 设置路径 ─────────────────────────
import subprocess, sys, shutil
from google.colab import drive, userdata

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Qwen-Reasoning'
os.chdir(PROJECT_DIR)
print(f'工作目录: {PROJECT_DIR}')

# 从 GitHub 拉取最新代码（确保 bug 修复已同步）
if os.path.isdir(f'{PROJECT_DIR}/.git'):
    subprocess.run(['git', 'pull', '--rebase'], check=False)
else:
    tmp_repo = '/tmp/_repo_tmp'
    if os.path.isdir(tmp_repo):
        shutil.rmtree(tmp_repo)
    r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/yukiiii0730/6000Q-QwenMiniReason.git', tmp_repo],
        capture_output=True, text=True)
    if r.returncode == 0:
        for item in os.listdir(tmp_repo):
            if item.startswith('.'):
                continue
            src = os.path.join(tmp_repo, item)
            dst = os.path.join(PROJECT_DIR, item)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
        shutil.rmtree(tmp_repo)
        print('✅ 代码已从 GitHub 同步到 Drive')
    else:
        print(f'⚠️ GitHub clone 失败: {r.stderr}')

# 设置 API key
try:
    os.environ['DASHSCOPE_API_KEY'] = userdata.get('DASHSCOPE_API_KEY')
    print('DASHSCOPE_API_KEY: OK')
except Exception:
    print('DASHSCOPE_API_KEY: missing（7B 评测将跳过）')

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN: OK')
except Exception:
    print('HF_TOKEN: missing')

sys.path.insert(0, f'{PROJECT_DIR}/eval')
sys.path.insert(0, f'{PROJECT_DIR}/scripts')
for d in ['logs', 'outputs']:
    os.makedirs(d, exist_ok=True)

print('✅ 环境就绪')


In [ ]:
# ── 0.3 检查模型 + NF4 检测 ─────────────────────────────────────
from model_loader import _has_quantized_weights, _find_adapter_dir

def read_adapter_base_model(adapter_path):
    """从 adapter_config.json 读取 base_model_name_or_path。"""
    cfg_path = f'{adapter_path}/adapter_config.json'
    if os.path.isfile(cfg_path):
        with open(cfg_path) as f:
            return json.load(f).get('base_model_name_or_path', '')
    return ''

def ensure_fp16_merged(merged_path, adapter_path_hint, label):
    """检测 NF4 并自动重合并为 fp16，返回可用路径。
    优先使用 adapter_path_hint，且会从 adapter_config 读取 base_model 传给 merge_lora.py。"""
    if not merged_path or not os.path.isfile(f'{merged_path}/config.json'):
        return merged_path
    if not _has_quantized_weights(merged_path):
        return merged_path

    # 确定 adapter 路径：优先用 hint，再 fallback 到自动检测
    adapter = None
    if adapter_path_hint and os.path.isfile(f'{adapter_path_hint}/adapter_config.json'):
        adapter = adapter_path_hint
    else:
        adapters = _find_adapter_dir(merged_path)
        if adapters:
            adapter = adapters[-1]  # 取最后一个（DPO 优先于 SFT）
    if not adapter:
        print(f'  ❌ {label}: NF4 且未找到 adapter')
        return merged_path

    # 从 adapter_config 读取 base_model
    base_model = read_adapter_base_model(adapter)

    fp16_path = merged_path.rstrip('/') + '_fp16'
    # 检查 fp16 缓存是否真的干净（之前可能生成了损坏的 fp16）
    if os.path.isfile(f'{fp16_path}/config.json') and not _has_quantized_weights(fp16_path):
        print(f'  {label}: fp16 版已存在且干净 → {fp16_path}')
        return fp16_path

    # 如果 fp16 目录存在但仍有 NF4，删除后重建
    if os.path.isdir(fp16_path):
        import shutil
        print(f'  {label}: fp16 目录存在但含 NF4 权重，删除重建...')
        shutil.rmtree(fp16_path)

    print(f'  {label}: NF4 → 重合并 fp16: {adapter} → {fp16_path}')
    cmd = ['python3', 'scripts/merge_lora.py',
           '--adapter_path', adapter, '--output_path', fp16_path]
    if base_model:
        cmd += ['--base_model', base_model]
        print(f'    base_model: {base_model}')
    subprocess.run(cmd, check=True)

    # 验证生成的 fp16 确实干净
    if os.path.isfile(f'{fp16_path}/config.json') and _has_quantized_weights(fp16_path):
        print(f'  ❌ {label}: merge_lora.py 输出仍含 NF4，可能需要升级 bitsandbytes')
        return fp16_path  # 仍返回，让后续报错提示

    print(f'  ✅ {label} fp16 完成: {fp16_path}')
    return fp16_path

# 检查各模型
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
G_A_DPO = 'outputs/group_a/dpo'
G_A_MERGED = 'outputs/group_a/merged'

# Group A DPO: 检查 adapter 和 merged
if os.path.isfile(f'{G_A_MERGED}/config.json'):
    print(f'Group A merged: OK ({G_A_MERGED})')
    G_A_MERGED = ensure_fp16_merged(G_A_MERGED, G_A_DPO, 'Group A DPO')
elif os.path.isfile(f'{G_A_DPO}/adapter_config.json'):
    print(f'Group A adapter 存在但 merged 不存在，需要先合并')
    print(f'  执行 merge_lora.py ...')
    os.makedirs(G_A_MERGED, exist_ok=True)
    g_a_sft_merged = 'outputs/group_a/sft_merged'
    subprocess.run([
        'python3', 'scripts/merge_lora.py',
        '--adapter_path', G_A_DPO,
        '--base_model', g_a_sft_merged,
        '--output_path', G_A_MERGED,
    ], check=True)
    print(f'  ✅ Group A DPO 合并完成: {G_A_MERGED}')
    G_A_MERGED = ensure_fp16_merged(G_A_MERGED, G_A_DPO, 'Group A DPO')
else:
    print(f'⚠️ Group A DPO 模型不存在（{G_A_DPO} 和 {G_A_MERGED}）')
    print(f'  请确认 Drive 上有 outputs/group_a/dpo/ 或 outputs/group_a/merged/')

# 检查已有结果
print('\n📋 已有结果:')
for f in ['logs/eval_supplement_1.5b_gsm8k.json', 'logs/eval_supplement_1.5b_math.json',
          'logs/eval_supplement_7b_gsm8k.json', 'logs/eval_supplement_7b_math.json',
          'logs/eval_supplement_group_a_dpo_gsm8k.json', 'logs/eval_supplement_group_a_dpo_math.json']:
    if os.path.isfile(f):
        d = json.load(open(f))
        acc = d.get('accuracy', 'N/A')
        total = d.get('total', 'N/A')
        print(f'  ✅ {f}: acc={acc}, n={total}')
    else:
        print(f'  ❌ {f}: 未跑')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E8: Targeted DPO 数据构建
# 策略：从 SFT 模型的 GSM8K + MATH badcase 中提取 DPO 对
#       chosen = gt_raw（正确解法）
#       rejected = pred_raw（模型的错误推理）
#       直接复用已有评测结果，无需重新推理
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E8: Targeted DPO 数据构建')
print('='*60)

TARGETED_DPO_DATA = 'data/processed/dpo_targeted_badcases.json'

# 快速路径
if os.path.isfile(TARGETED_DPO_DATA):
    d = json.load(open(TARGETED_DPO_DATA))
    n = sum(1 for x in d if x.get('chosen') and x.get('rejected'))
    print(f'  ✅ 已有 Targeted DPO 数据: {n} 有效对')
else:
    import re

    # ── 加载 badcase 源 ──
    # 优先用 logs 3/（最新、含 error classification）
    gsm8k_src = 'logs 3/v1/gsm8k_b_sft_badcases.jsonl'   # 439 条
    math_src = 'logs 3/math_sft_badcases.jsonl'            # 112 条

    # fallback 到 logs 2/
    if not os.path.isfile(gsm8k_src):
        gsm8k_src = 'logs 2/gsm8k_sft_badcases.jsonl'
    if not os.path.isfile(math_src):
        math_src = 'logs 2/math_sft_badcases.jsonl'

    def load_jsonl(path):
        if not os.path.isfile(path):
            print(f'  ⚠️ 文件不存在: {path}')
            return []
        with open(path, encoding='utf-8') as f:
            return [json.loads(line) for line in f if line.strip()]

    gsm8k_bad = load_jsonl(gsm8k_src)
    math_bad = load_jsonl(math_src)
    print(f'  GSM8K badcases: {len(gsm8k_bad)} 条 ({gsm8k_src})')
    print(f'  MATH  badcases: {len(math_bad)} 条 ({math_src})')

    # ── 构造 DPO 对 ──
    SYSTEM_PROMPT = '请先进行清晰的逐步推理，再给出最终答案。'
    completed = []
    seen_questions = set()

    def build_chat_prompt(question):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': question},
        ]
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return f'{SYSTEM_PROMPT}\n\n{question}'

    # 确保 tokenizer 可用（从 E5 或重新加载）
    if 'tokenizer' not in dir():
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

    # GSM8K badcases
    gsm8k_added = 0
    for i, item in enumerate(gsm8k_bad):
        q = item.get('question', '')
        gt_raw = item.get('gt_raw', '')
        pred_raw = item.get('pred_raw', '')
        if not q or not gt_raw or not pred_raw:
            continue
        if q in seen_questions:
            continue
        seen_questions.add(q)

        prompt = build_chat_prompt(q)
        # gt_raw 格式: "step1...\n#### answer" → 去掉计算标记，保留推理
        chosen = gt_raw.replace('<<', '').replace('>>', '')
        completed.append({
            'prompt': prompt,
            'chosen': chosen,
            'rejected': pred_raw.strip(),
            '_source': 'gsm8k_badcase',
        })
        gsm8k_added += 1
        if (i + 1) % 100 == 0:
            print(f'    GSM8K [{i+1}/{len(gsm8k_bad)}] 已构建 {gsm8k_added} 对')
    print(f'    GSM8K 完成: {gsm8k_added} 对')

    # MATH badcases
    math_added = 0
    for i, item in enumerate(math_bad):
        q = item.get('problem', '') or item.get('question', '')
        gt_raw = item.get('gt_raw', '')
        pred_raw = item.get('pred_raw', '')
        if not q or not gt_raw or not pred_raw:
            continue
        if q in seen_questions:
            continue
        seen_questions.add(q)

        prompt = build_chat_prompt(q)
        # MATH gt_raw 已是完整 LaTeX 推理链，直接用
        completed.append({
            'prompt': prompt,
            'chosen': gt_raw.strip(),
            'rejected': pred_raw.strip(),
            '_source': 'math_badcase',
        })
        math_added += 1
        if (i + 1) % 50 == 0:
            print(f'    MATH  [{i+1}/{len(math_bad)}] 已构建 {math_added} 对')
    print(f'    MATH  完成: {math_added} 对')

    print(f'\n  构造完成:')
    print(f'    GSM8K → {gsm8k_added} 对')
    print(f'    MATH  → {math_added} 对')
    print(f'    总计  → {len(completed)} 对 (去重后)')

    # 保存
    with open(TARGETED_DPO_DATA, 'w', encoding='utf-8') as f:
        json.dump(completed, f, ensure_ascii=False, indent=2)
    print(f'  已保存: {TARGETED_DPO_DATA}')

    # 质量抽检：打印 1 条样例
    if completed:
        sample = completed[0]
        print(f'\n  ── 样例 ──')
        print(f'  prompt (前80字符): {sample["prompt"][:80]}...')
        print(f'  chosen (前80字符): {sample["chosen"][:80]}...')
        print(f'  rejected (前80字符): {sample["rejected"][:80]}...')

print('\nE8 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E9: Targeted DPO 训练
# 策略：用 SFT 模型的 badcase 直接训练 DPO，针对弱点优化
# Base: outputs/sft_merged
# Data: data/processed/dpo_targeted_badcases.json（E8 产出）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E9: Targeted DPO 训练')
print('='*60)

G_T_DPO = 'outputs/group_targeted/dpo'
G_T_MERGED = 'outputs/group_targeted/merged'
SFT_BASE = 'outputs/sft_merged'
TARGETED_DPO_DATA = 'data/processed/dpo_targeted_badcases.json'

if not os.path.isfile(TARGETED_DPO_DATA):
    print(f'❌ Targeted DPO 数据不存在: {TARGETED_DPO_DATA}')
    print(f'  请先运行 E8')
elif not os.path.isfile(f'{SFT_BASE}/config.json'):
    print(f'❌ SFT base 模型不存在: {SFT_BASE}/config.json')
else:
    data = json.load(open(TARGETED_DPO_DATA))
    n = sum(1 for x in data if x.get('chosen') and x.get('rejected'))
    print(f'  Targeted DPO 数据: {len(data)} 条, 有效对: {n}')

    # 检查是否已训练
    if os.path.isfile(f'{G_T_MERGED}/config.json'):
        print(f'✅ Targeted DPO merged 已存在: {G_T_MERGED}')
    elif os.path.isfile(f'{G_T_DPO}/adapter_config.json'):
        print(f'✅ Targeted DPO adapter 已存在: {G_T_DPO}')
    else:
        # 写入配置
        g_t_config = 'config/dpo_targeted.yaml'
        import yaml
        gt_cfg = {
            'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
            'base_adapter_path': SFT_BASE,
            'output_dir': G_T_DPO,
            'max_seq_length': 2048,
            'load_in_4bit': True,
            'seed': 42,
            'beta': 0.1,
            'loss_type': 'sigmoid',
            'dataset': {
                'name': 'local',
                'split': 'train',
                'max_samples': 1500,
            },
            'lora': {
                'use_dora': True,
                'r': 16,
                'alpha': 32,
                'dropout': 0.0,
                'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                   'gate_proj', 'up_proj', 'down_proj'],
            },
            'train': {
                'per_device_train_batch_size': 1,
                'gradient_accumulation_steps': 16,
                'warmup_steps': 30,
                'max_steps': 400,
                'learning_rate': 1e-5,
                'logging_steps': 10,
                'save_steps': 100,
                'eval_steps': 100,
                'weight_decay': 0.0,
                'lr_scheduler_type': 'cosine',
                'optim': 'paged_adamw_8bit',
                'fp16': False,
                'bf16': True,
                'dataloader_num_workers': 4,
                'dataloader_pin_memory': True,
            },
            'dataset_path': TARGETED_DPO_DATA,
        }
        with open(g_t_config, 'w') as f:
            yaml.dump(gt_cfg, f, default_flow_style=False, allow_unicode=True)
        print(f'  配置已写入: {g_t_config}')
        print(f'  数据: {TARGETED_DPO_DATA} ({n} 对)')
        print(f'  输出: {G_T_DPO}')
        print(f'  max_steps: 400（数据量较小，减少步数防过拟合）')

        # 运行训练
        run_eval([
            'python3', '-u', 'scripts/dpo_train.py',
            '--config', g_t_config,
        ], 'Targeted DPO 训练')

    # 合并 LoRA → merged
    if os.path.isfile(f'{G_T_DPO}/adapter_config.json') and not os.path.isfile(f'{G_T_MERGED}/config.json'):
        # 先确保 base model 是 fp16（sft_merged 可能含 NF4 权重）
        merge_base = ensure_fp16_merged(SFT_BASE, 'outputs/sft', 'SFT Base')
        print(f'\n  合并 Targeted DPO LoRA → {G_T_MERGED}...')
        print(f'  base model: {merge_base}')
        os.makedirs(G_T_MERGED, exist_ok=True)
        r = subprocess.run([
            'python3', 'scripts/merge_lora.py',
            '--adapter_path', G_T_DPO,
            '--base_model', merge_base,
            '--output_path', G_T_MERGED,
        ], capture_output=True, text=True)
        if r.returncode != 0:
            print(f'  ❌ merge_lora.py 失败 (exit {r.returncode})')
            print(f'  stdout: {r.stdout[-500:]}')
            print(f'  stderr: {r.stderr[-500:]}')
        else:
            print(f'  ✅ Targeted DPO 合并完成: {G_T_MERGED}')

    # NF4 检测
    if os.path.isfile(f'{G_T_MERGED}/config.json'):
        G_T_MERGED = ensure_fp16_merged(G_T_MERGED, G_T_DPO, 'Targeted DPO')
        print(f'  ✅ Targeted DPO 模型就绪: {G_T_MERGED}')
    else:
        print(f'  ❌ Targeted DPO 模型不存在（训练可能失败）')

print('\nE9 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E10: Targeted DPO 评测（GSM8K + MATH-500，n=200）
# 模型：outputs/group_targeted/merged（SFT + Badcase-Targeted DPO）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# 确保 G_T_MERGED 变量存在
if 'G_T_MERGED' not in dir() or not G_T_MERGED:
    G_T_MERGED = 'outputs/group_targeted/merged'

if not os.path.isfile(f'{G_T_MERGED}/config.json'):
    print('⚠️ Targeted DPO 模型不存在，跳过 E10')
    print(f'  检查: {G_T_MERGED}/config.json')
else:
    print('\n' + '='*60)
    print(f'  E10: Targeted DPO 评测 (模型: {G_T_MERGED})')
    print('='*60)

    # GSM8K
    gsm_gt_out = 'logs/eval_supplement_targeted_dpo_gsm8k.json'
    if not is_eval_complete(gsm_gt_out, EVAL_N):
        run_eval([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', G_T_MERGED,
            '--max_samples', EVAL_N,
            '--output', gsm_gt_out,
        ] + BIT, 'Targeted DPO GSM8K')
    print_result(gsm_gt_out, 'Targeted DPO GSM8K')

    # MATH-500
    math_gt_out = 'logs/eval_supplement_targeted_dpo_math.json'
    if not is_eval_complete(math_gt_out, EVAL_N):
        run_eval([
            'python3', 'eval/math_eval.py',
            '--model_path', G_T_MERGED,
            '--max_samples', EVAL_N,
            '--output', math_gt_out,
        ] + BIT, 'Targeted DPO MATH')
    print_result(math_gt_out, 'Targeted DPO MATH')

    print('\nE10 完成')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DIAG: Targeted DPO 诊断
# 排查 25% accuracy 的原因：merge 问题 vs DPO 训练问题
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from model_loader import _has_quantized_weights

print('\n' + '='*60)
print('  DIAG: Targeted DPO 诊断')
print('='*60)

G_T_DPO = 'outputs/group_targeted/dpo'
G_T_MERGED = 'outputs/group_targeted/merged'
SFT_FP16 = 'outputs/sft_merged_fp16'
SFT_RAW = 'outputs/sft_merged'

# ── 1. 检查 adapter 的 base_model 指向 ──
print('\n── 1. Adapter base_model 检查 ──')
adapter_cfg_path = f'{G_T_DPO}/adapter_config.json'
if os.path.isfile(adapter_cfg_path):
    import json as _json
    with open(adapter_cfg_path) as f:
        acfg = _json.load(f)
    base_ref = acfg.get('base_model_name_or_path', '(未设置)')
    print(f'  adapter base_model: {base_ref}')
    print(f'  SFT_RAW 路径:      {os.path.abspath(SFT_RAW)}')
    print(f'  SFT_FP16 路径:     {os.path.abspath(SFT_FP16)}')
    if 'sft' in base_ref.lower():
        print(f'  ✅ base_model 指向 SFT 模型')
    else:
        print(f'  ⚠️ base_model 可能不匹配！')
else:
    print(f'  ❌ adapter_config.json 不存在')

# ── 2. 检查 merged 模型是否含 NF4 ──
print('\n── 2. Merged 模型权重检查 ──')
if os.path.isfile(f'{G_T_MERGED}/model.safetensors'):
    nf4_in_merged = _has_quantized_weights(G_T_MERGED)
    if nf4_in_merged:
        print(f'  ❌ merged 模型仍含 NF4 权重！merge 反量化不完整')
    else:
        print(f'  ✅ merged 模型权重是 fp16')
    
    # 检查权重 shape
    from safetensors import safe_open
    f = safe_open(f'{G_T_MERGED}/model.safetensors', framework='pt')
    keys = list(f.keys())[:5]
    for k in keys:
        t = f.get_tensor(k)
        print(f'  {k}: dtype={t.dtype} shape={list(t.shape)}')
else:
    print(f'  ❌ merged safetensors 不存在')

# ── 3. 对比测试：SFT-only vs Adapter-direct vs Merged ──
print('\n── 3. 三模型对比测试（5 道 GSM8K 样题）──')

# 固定测试题
test_questions = [
    "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
    "A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?",
    "Josh decides to try flipping a house. He buys a house for $80,000 and then puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?",
    "Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables. She gives the chickens their feed in three separate meals. She filled her scoop with four cups of feed for the first meal of the day. During the second meal, she gave the chickens 2 more cups than she gave them for the first meal. How many scoops of feed does she need for the final meal of the day?",
    "Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?",
]
test_answers = ['18', '3', '70000', '2', '64']

def quick_test(model, tokenizer, label, n=5):
    correct = 0
    for q, gt in zip(test_questions[:n], test_answers[:n]):
        messages = [
            {'role': 'system', 'content': '请先进行清晰的逐步推理，再给出最终答案。'},
            {'role': 'user', 'content': q},
        ]
        try:
            input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except:
            input_text = f"请先进行清晰的逐步推理，再给出最终答案。\n\n{q}"
        inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        # 提取答案（与 eval 脚本一致）
        import re
        pred = ''
        boxed = re.findall(r'\\boxed\{\s*(-?\d+(?:\.\d+)?)\s*\}', resp)
        if boxed:
            pred = boxed[-1]
        if not pred:
            nums = re.findall(r'####\s*(-?\d+(?:\.\d+)?)', resp)
            if nums:
                pred = nums[-1]
        if not pred:
            nums = re.findall(r'-?\d+(?:\.\d+)?', resp[-300:])
            pred = nums[-1] if nums else ''
        # 归一化：18.0 → 18，去除尾部句点
        pred = pred.rstrip('.')
        if pred and '.' in pred:
            try:
                f = float(pred)
                if f == int(f):
                    pred = str(int(f))
            except ValueError:
                pass
        ok = pred.strip() == gt.strip()
        correct += int(ok)
        print(f'    pred={pred:10s} gt={gt:5s} {"✅" if ok else "❌"}')
    print(f'  {label}: {correct}/{n} correct')
    return correct

tokenizer = AutoTokenizer.from_pretrained(SFT_RAW, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3a. SFT-only (baseline)
print('\n  ── 3a. SFT-only (baseline) ──')
if os.path.isdir(SFT_FP16):
    sft_model = AutoModelForCausalLM.from_pretrained(
        SFT_FP16, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
    sft_score = quick_test(sft_model, tokenizer, 'SFT-only')
    del sft_model
    torch.cuda.empty_cache()
else:
    print(f'  ⚠️ {SFT_FP16} 不存在，跳过 SFT baseline')

# 3b. Adapter-direct (不 merge)
print('\n  ── 3b. DPO Adapter 直接推理（不 merge）──')
if os.path.isdir(SFT_FP16) and os.path.isfile(adapter_cfg_path):
    from peft import PeftModel
    adapter_model = AutoModelForCausalLM.from_pretrained(
        SFT_FP16, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
    adapter_model = PeftModel.from_pretrained(adapter_model, G_T_DPO)
    adapter_score = quick_test(adapter_model, tokenizer, 'Adapter-direct')
    del adapter_model
    torch.cuda.empty_cache()
else:
    print(f'  ⚠️ 缺少 SFT fp16 或 adapter，跳过')

# 3c. Merged 模型
print('\n  ── 3c. Merged 模型 ──')
if os.path.isfile(f'{G_T_MERGED}/config.json'):
    merged_model = AutoModelForCausalLM.from_pretrained(
        G_T_MERGED, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
    merged_score = quick_test(merged_model, tokenizer, 'Merged')
    del merged_model
    torch.cuda.empty_cache()
else:
    print(f'  ⚠️ {G_T_MERGED} 不存在，跳过')

# ── 4. 结论 ──
print('\n── 4. 诊断结论 ──')
print('  如果 SFT-only 和 Adapter-direct 分数接近，但 Merged 分数低：')
print('    → 问题在 merge 过程，不是 DPO 训练的问题')
print('  如果 Adapter-direct 分数也很低：')
print('    → DPO 训练本身有问题（过拟合/数据质量）')
print('  如果 SFT-only 分数就低：')
print('    → SFT base 模型本身有问题')

print('\nDIAG 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E11: Teacher SFT 数据准备
# 将 teacher DPO 数据转换为 SFT 格式（prompt + chosen）
# 参考 DeepSeek-R1-Distill: SFT 蒸馏 > DPO 对小模型
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E11: Teacher SFT 数据准备')
print('='*60)

TEACHER_DPO_DATA = 'data/processed/dpo_teacher_round_1.json'
TEACHER_SFT_DATA = 'data/processed/sft_teacher_gsm8k.json'

if not os.path.isfile(TEACHER_DPO_DATA):
    print(f'❌ Teacher DPO 数据不存在: {TEACHER_DPO_DATA}')
    print(f'  请先运行 build_teacher_dpo.py')
else:
    # 加载原始 teacher DPO 数据
    with open(TEACHER_DPO_DATA) as f:
        dpo_data = json.load(f)
    
    print(f'  源数据: {len(dpo_data)} 条 ({TEACHER_DPO_DATA})')
    
    # 转换为 SFT 格式：只保留 prompt + chosen（丢弃 rejected）
    sft_data = []
    skipped = 0
    for i, item in enumerate(dpo_data):
        prompt = item.get('prompt', '')
        chosen = item.get('chosen', '')
        if prompt and chosen:
            sft_data.append({
                'instruction': prompt,
                'output': chosen,
            })
        else:
            skipped += 1
        if (i + 1) % 500 == 0:
            print(f'    [{i+1}/{len(dpo_data)}] 已转换 {len(sft_data)} 条, 跳过 {skipped} 条')
    
    os.makedirs(os.path.dirname(TEACHER_SFT_DATA), exist_ok=True)
    with open(TEACHER_SFT_DATA, 'w') as f:
        json.dump(sft_data, f, ensure_ascii=False, indent=2)
    
    print(f'    转换完成: {len(sft_data)} 条, 跳过 {skipped} 条')
    print(f'  输出: {TEACHER_SFT_DATA}')
    print(f'  格式: instruction + output（无 rejected）')
    print(f'  ✅ 数据准备完成')

print('\nE11 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E12: Teacher SFT 训练
# 参考 DeepSeek-R1-Distill: 用教师模型的推理轨迹做 SFT
# 参数设计思考：
#   - LoRA（非 DoRA）：1500 条高质量数据，LoRA 更简洁
#   - lr=5e-5：LoRA SFT 常用 1e-4~2e-4，但数据量小故保守取 5e-5
#   - packing=False：teacher CoT 长度 ~3k 字符，packing 会截断推理链
#   - 5 epochs / 468 steps：1500 条数据量小，多跑几轮充分学习
#   - gradient_checkpointing：不开 packing 后显存增加，需要开启
# Base: Qwen2.5-1.5B-Instruct（从头 SFT，不用五段课程）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E12: Teacher SFT 训练')
print('='*60)

G_TS_SFT = 'outputs/group_teacher_sft/sft'
G_TS_MERGED = 'outputs/group_teacher_sft/merged'
TEACHER_SFT_DATA = 'data/processed/sft_teacher_gsm8k.json'

if not os.path.isfile(TEACHER_SFT_DATA):
    print(f'❌ Teacher SFT 数据不存在: {TEACHER_SFT_DATA}')
    print(f'  请先运行 E11')
else:
    with open(TEACHER_SFT_DATA) as f:
        sft_data = json.load(f)
    n_data = len(sft_data)
    
    # 参数设计：
    # 1500 条, effective BS=16, 5 epochs → steps = 5*1500/16 ≈ 468
    # packing=False → 不截断 teacher CoT 推理链
    # lr=5e-5 → LoRA SFT 常用范围，数据量小故保守
    max_steps = 470
    warmup_steps = 47  # 10%
    
    print(f'  数据: {n_data} 条')
    print(f'  训练: {max_steps} steps, warmup={warmup_steps}, lr=5e-5')
    print(f'  PEFT: LoRA (非 DoRA), packing=False')
    print(f'  参考: DeepSeek-R1-Distill 策略（教师推理轨迹 SFT）')
    
    # 检查是否已训练
    if os.path.isfile(f'{G_TS_MERGED}/config.json'):
        print(f'✅ Teacher SFT merged 已存在: {G_TS_MERGED}')
    elif os.path.isfile(f'{G_TS_SFT}/adapter_config.json'):
        print(f'✅ Teacher SFT adapter 已存在: {G_TS_SFT}')
    else:
        import yaml
        ts_config_path = 'config/sft_teacher.yaml'
        ts_cfg = {
            'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
            'output_dir': G_TS_SFT,
            'max_seq_length': 2048,
            'load_in_4bit': True,
            'seed': 42,
            'dataset': {
                'name': 'local',
                'path': TEACHER_SFT_DATA,
                'split': 'train',
                'max_samples': n_data,
            },
            'lora': {
                'use_dora': False,   # LoRA，非 DoRA
                'r': 16,
                'alpha': 32,
                'dropout': 0.05,
                'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                   'gate_proj', 'up_proj', 'down_proj'],
            },
            'train': {
                'per_device_train_batch_size': 2,
                'gradient_accumulation_steps': 8,
                'warmup_steps': warmup_steps,
                'max_steps': max_steps,
                'learning_rate': 5e-5,
                'logging_steps': 10,
                'save_steps': 100,
                'weight_decay': 0.01,
                'lr_scheduler_type': 'cosine',
                'optim': 'paged_adamw_8bit',
                'fp16': False,
                'bf16': True,
                'packing': False,    # 不 packing，保留完整 teacher CoT
                'gradient_checkpointing': True,  # 不 packing 后显存增加，开启节省
                'num_train_epochs': 5,
            },
        }
        with open(ts_config_path, 'w') as f:
            yaml.dump(ts_cfg, f, default_flow_style=False, allow_unicode=True)
        print(f'  配置已写入: {ts_config_path}')

        # 运行 SFT 训练
        run_eval([
            'python3', '-u', 'scripts/sft_train.py',
            '--config', ts_config_path,
        ], 'Teacher SFT 训练')

    # 合并 LoRA → merged
    if os.path.isfile(f'{G_TS_SFT}/adapter_config.json') and not os.path.isfile(f'{G_TS_MERGED}/config.json'):
        print(f'\n  合并 Teacher SFT LoRA → {G_TS_MERGED}...')
        os.makedirs(G_TS_MERGED, exist_ok=True)
        r = subprocess.run([
            'python3', 'scripts/merge_lora.py',
            '--adapter_path', G_TS_SFT,
            '--base_model', 'Qwen/Qwen2.5-1.5B-Instruct',
            '--output_path', G_TS_MERGED,
        ], capture_output=True, text=True)
        if r.returncode != 0:
            print(f'  ❌ merge_lora.py 失败 (exit {r.returncode})')
            print(f'  stdout: {r.stdout[-500:]}')
            print(f'  stderr: {r.stderr[-500:]}')
        else:
            print(f'  ✅ Teacher SFT 合并完成: {G_TS_MERGED}')

    # NF4 检测
    if os.path.isfile(f'{G_TS_MERGED}/config.json'):
        G_TS_MERGED = ensure_fp16_merged(G_TS_MERGED, G_TS_SFT, 'Teacher SFT')
        print(f'  ✅ Teacher SFT 模型就绪: {G_TS_MERGED}')
    else:
        print(f'  ❌ Teacher SFT 模型不存在（训练可能失败）')

print('\nE12 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E13: Teacher SFT 评测（GSM8K + MATH-500，n=200）
# 模型：outputs/group_teacher_sft/merged（Teacher 蒸馏 SFT）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# 确保 G_TS_MERGED 变量存在
if 'G_TS_MERGED' not in dir() or not G_TS_MERGED:
    G_TS_MERGED = 'outputs/group_teacher_sft/merged'

if not os.path.isfile(f'{G_TS_MERGED}/config.json'):
    print('⚠️ Teacher SFT 模型不存在，跳过 E13')
    print(f'  检查: {G_TS_MERGED}/config.json')
else:
    print('\n' + '='*60)
    print(f'  E13: Teacher SFT 评测 (模型: {G_TS_MERGED})')
    print('='*60)

    # GSM8K
    gsm_ts_out = 'logs/eval_supplement_teacher_sft_gsm8k.json'
    if not is_eval_complete(gsm_ts_out, EVAL_N):
        run_eval([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', G_TS_MERGED,
            '--max_samples', EVAL_N,
            '--output', gsm_ts_out,
        ] + BIT, 'Teacher SFT GSM8K')
    print_result(gsm_ts_out, 'Teacher SFT GSM8K')

    # MATH-500
    math_ts_out = 'logs/eval_supplement_teacher_sft_math.json'
    if not is_eval_complete(math_ts_out, EVAL_N):
        run_eval([
            'python3', 'eval/math_eval.py',
            '--model_path', G_TS_MERGED,
            '--max_samples', EVAL_N,
            '--output', math_ts_out,
        ] + BIT, 'Teacher SFT MATH')
    print_result(math_ts_out, 'Teacher SFT MATH')

    print('\nE13 完成')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E5: Group C Teacher DPO 数据补全
# 策略：用 SFT 模型对 1500 道 teacher 题目生成答案，
#       答错的作为 rejected，与 teacher chosen 组成 DPO 对
#       chosen = Qwen3-235B-Thinking 的 CoT（已存在）
#       rejected = 1.5B SFT 模型的真实错误答案（本 cell 生成）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

from model_loader import load_model_and_tokenizer

print('\n' + '='*60)
print('  E5: Group C Teacher DPO 数据补全')
print('='*60)

TEACHER_RAW = 'data/processed/dpo_teacher_round_1.json'
G_C_DPO_DATA = 'data/processed/dpo_teacher_group_c.json'

# 快速路径：已完成的数据
if os.path.isfile(G_C_DPO_DATA):
    d = json.load(open(G_C_DPO_DATA))
    n = sum(1 for x in d if x.get('chosen') and x.get('rejected'))
    print(f'  ✅ 已有 Group C DPO 数据: {n} 有效对')
else:
    teacher = json.load(open(TEACHER_RAW))
    print(f'  Teacher 数据: {len(teacher)} 条 (chosen 有, rejected 空)')

    # 加载 SFT 模型（通过 model_loader 处理 NF4 检测 + adapter 回退）
    sft_model = 'outputs/sft_merged'
    if not os.path.isfile(f'{sft_model}/config.json'):
        print(f'  ❌ SFT 模型不存在: {sft_model}')
        print(f'  请确认 Drive 上有 outputs/sft_merged/')
    else:
        import torch

        print(f'  加载 SFT 模型: {sft_model} ...')
        model, tokenizer = load_model_and_tokenizer(sft_model, load_in_4bit=False)
        model.eval()
        print(f'  ✅ 模型加载完成')

        # 答案提取（与 math_eval.py / gsm8k_eval.py 一致）
        import re
        _BOXED_RE = re.compile(r'\\boxed\s*\{')

        def extract_boxed(text):
            if not text:
                return ''
            last = ''
            for m in _BOXED_RE.finditer(text):
                i = m.end()
                depth = 1
                out = []
                while i < len(text) and depth > 0:
                    c = text[i]
                    if c == '{':
                        depth += 1
                        out.append(c)
                    elif c == '}':
                        depth -= 1
                        if depth == 0:
                            break
                        out.append(c)
                    else:
                        out.append(c)
                    i += 1
                last = ''.join(out).strip()
            return last

        def extract_answer(text):
            if not text:
                return ''
            boxed = extract_boxed(text)
            if boxed:
                return boxed
            m = re.search(r'####\s*(.+)', text)
            if m:
                return m.group(1).strip()
            nums = re.findall(r'-?\d+\.?\d*', text)
            return nums[-1] if nums else ''

        def _strip_string(s):
            if s is None:
                return ''
            s = str(s).strip()
            s = s.replace('$', '').replace(' ', '').replace('\n', '')
            if '=' in s and len(s.split('=')[-1]) > 0:
                s = s.split('=')[-1]
            s = re.sub(r'\\text\{(.*?)\}', r'\1', s)
            s = s.replace('\\dfrac', '\\frac').replace('\\tfrac', '\\frac')
            s = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', s)
            s = re.sub(r'\\sqrt\{([^}]+)\}', r'sqrt(\1)', s)
            s = s.replace('^\\circ', '').replace('\\circ', '')
            s = re.sub(r'(\d),(\d)', r'\1\2', s)
            s = s.replace('\\left', '').replace('\\right', '')
            s = s.rstrip('.')
            return s

        def _to_float(s):
            try:
                return float(s)
            except:
                m = re.match(r'^(-?)(\d+)\s*/\s*(\d+)$', s)
                if m:
                    sign = -1 if m.group(1) == '-' else 1
                    num, den = int(m.group(2)), int(m.group(3))
                    if den != 0:
                        return sign * num / den
            return None

        def is_equiv(pred, gt):
            if pred is None or gt is None:
                return False
            p = _strip_string(pred)
            g = _strip_string(gt)
            if not p or not g:
                return False
            if p == g:
                return True
            pf, gf = _to_float(p), _to_float(g)
            if pf is not None and gf is not None:
                if abs(pf - gf) < 1e-4 or (gf != 0 and abs((pf - gf) / gf) < 1e-4):
                    return True
            return False

        # 逐题生成 rejected
        print(f'  开始推理 {len(teacher)} 题...')
        completed = []
        n_correct = 0
        n_wrong = 0
        n_skip = 0
        for i, item in enumerate(teacher):
            prompt = item['prompt']
            chosen = item['chosen']
            gt = extract_answer(chosen)
            if not gt:
                n_skip += 1
                continue

            messages = [{'role': 'user', 'content': prompt}]
            input_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=512, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id)
            resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                    skip_special_tokens=True)

            pred = extract_answer(resp)
            if is_equiv(pred, gt):
                n_correct += 1
            else:
                n_wrong += 1
                completed.append({
                    'prompt': prompt,
                    'chosen': chosen,
                    'rejected': resp,
                    '_source': 'gsm8k',
                })

            if (i + 1) % 50 == 0 or i == len(teacher) - 1:
                print(f'    [{i+1}/{len(teacher)}] 答对={n_correct} 答错={n_wrong} 跳过={n_skip}  DPO对={len(completed)}')

        with open(G_C_DPO_DATA, 'w') as f:
            json.dump(completed, f, ensure_ascii=False, indent=2)
        total_attempted = n_correct + n_wrong
        print(f'\n  ✅ Group C DPO 数据: {len(completed)} 对')
        print(f'  答对: {n_correct}, 答错: {n_wrong}, 跳过: {n_skip}')
        if total_attempted > 0:
            print(f'  答错率: {n_wrong/total_attempted*100:.1f}%')
        print(f'  已保存: {G_C_DPO_DATA}')

print('\nE5 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E6: Group C DPO 训练（DoRA + 五段课程 SFT + Teacher DPO）
# Base: outputs/sft_merged（与 Group B 相同的 SFT 基座）
# Data: data/processed/dpo_teacher_group_c.json（E5 产出）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E6: Group C DPO 训练')
print('='*60)

# 路径定义
G_C_DPO = 'outputs/group_c/dpo'
G_C_MERGED = 'outputs/group_c/merged'
SFT_BASE = 'outputs/sft_merged'

# 确定 teacher 数据路径（E5 产出）
teacher_data_path = 'data/processed/dpo_teacher_group_c.json'
if not os.path.isfile(teacher_data_path):
    print(f'❌ Group C DPO 数据不存在: {teacher_data_path}')
    print(f'  请先运行 E5 生成数据')
else:
    data = json.load(open(teacher_data_path))
    n = sum(1 for x in data if x.get('chosen') and x.get('rejected'))
    print(f'  Group C DPO 数据: {len(data)} 条, 有效对: {n}')

    # 检查 SFT base
    if not os.path.isfile(f'{SFT_BASE}/config.json'):
        print(f'❌ SFT base 模型不存在: {SFT_BASE}/config.json')
    else:
        # 检查是否已训练完成
        if os.path.isfile(f'{G_C_MERGED}/config.json'):
            print(f'✅ Group C merged 模型已存在: {G_C_MERGED}')
        elif os.path.isfile(f'{G_C_DPO}/adapter_config.json'):
            print(f'✅ Group C DPO adapter 已存在: {G_C_DPO}')
        else:
            # 写入 Group C 专用 DPO 配置
            g_c_config = 'config/dpo_group_c.yaml'
            import yaml
            gc_cfg = {
                'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
                'base_adapter_path': SFT_BASE,
                'output_dir': G_C_DPO,
                'max_seq_length': 2048,
                'load_in_4bit': True,
                'seed': 42,
                'beta': 0.1,
                'loss_type': 'sigmoid',
                'dataset': {
                    'name': 'local',
                    'split': 'train',
                    'max_samples': 1500,
                },
                'lora': {
                    'use_dora': True,
                    'r': 16,
                    'alpha': 32,
                    'dropout': 0.0,
                    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                       'gate_proj', 'up_proj', 'down_proj'],
                },
                'train': {
                    'per_device_train_batch_size': 1,
                    'gradient_accumulation_steps': 16,
                    'warmup_steps': 50,
                    'max_steps': 600,
                    'learning_rate': 1e-5,
                    'logging_steps': 10,
                    'save_steps': 100,
                    'eval_steps': 100,
                    'weight_decay': 0.0,
                    'lr_scheduler_type': 'cosine',
                    'optim': 'paged_adamw_8bit',
                    'fp16': False,
                    'bf16': True,
                    'dataloader_num_workers': 4,
                    'dataloader_pin_memory': True,
                },
                'dataset_path': teacher_data_path,
            }
            with open(g_c_config, 'w') as f:
                yaml.dump(gc_cfg, f, default_flow_style=False, allow_unicode=True)
            print(f'  配置已写入: {g_c_config}')
            print(f'  数据: {teacher_data_path}')
            print(f'  输出: {G_C_DPO}')

            # 运行 DPO 训练
            run_eval([
                'python3', '-u', 'scripts/dpo_train.py',
                '--config', g_c_config,
            ], 'Group C DPO 训练')

        # 合并 LoRA → merged 模型
        if os.path.isfile(f'{G_C_DPO}/adapter_config.json') and not os.path.isfile(f'{G_C_MERGED}/config.json'):
            # 先确保 base model 是 fp16（sft_merged 可能含 NF4 权重）
            merge_base = ensure_fp16_merged(SFT_BASE, 'outputs/sft', 'SFT Base')
            print(f'\n  合并 Group C DPO LoRA → {G_C_MERGED}...')
            print(f'  base model: {merge_base}')
            os.makedirs(G_C_MERGED, exist_ok=True)
            r = subprocess.run([
                'python3', 'scripts/merge_lora.py',
                '--adapter_path', G_C_DPO,
                '--base_model', merge_base,
                '--output_path', G_C_MERGED,
            ], capture_output=True, text=True)
            if r.returncode != 0:
                print(f'  ❌ merge_lora.py 失败 (exit {r.returncode})')
                print(f'  stdout: {r.stdout[-500:]}')
                print(f'  stderr: {r.stderr[-500:]}')
            else:
                print(f'  ✅ Group C 合并完成: {G_C_MERGED}')

        # NF4 检测
        if os.path.isfile(f'{G_C_MERGED}/config.json'):
            G_C_MERGED = ensure_fp16_merged(G_C_MERGED, G_C_DPO, 'Group C DPO')
            print(f'  ✅ Group C 模型就绪: {G_C_MERGED}')
        else:
            print(f'  ❌ Group C 模型不存在（训练可能失败）')

    print('\nE6 完成')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E7: Group C 评测（GSM8K + MATH-500，n=200）
# 模型：outputs/group_c/merged（DoRA + 五段课程 SFT + Teacher DPO）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# 确保 G_C_MERGED 变量存在
if 'G_C_MERGED' not in dir() or not G_C_MERGED:
    G_C_MERGED = 'outputs/group_c/merged'

if not os.path.isfile(f'{G_C_MERGED}/config.json'):
    print('⚠️ Group C DPO 模型不存在，跳过 E7')
    print(f'  检查: {G_C_MERGED}/config.json')
else:
    print('\n' + '='*60)
    print(f'  E7: Group C DPO 评测 (模型: {G_C_MERGED})')
    print('='*60)

    # GSM8K
    gsm_gc_out = 'logs/eval_supplement_group_c_dpo_gsm8k.json'
    if not is_eval_complete(gsm_gc_out, EVAL_N):
        run_eval([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', G_C_MERGED,
            '--max_samples', EVAL_N,
            '--output', gsm_gc_out,
        ] + BIT, 'Group C DPO GSM8K')
    print_result(gsm_gc_out, 'Group C DPO GSM8K')

    # MATH-500
    math_gc_out = 'logs/eval_supplement_group_c_dpo_math.json'
    if not is_eval_complete(math_gc_out, EVAL_N):
        run_eval([
            'python3', 'eval/math_eval.py',
            '--model_path', G_C_MERGED,
            '--max_samples', EVAL_N,
            '--output', math_gc_out,
        ] + BIT, 'Group C DPO MATH')
    print_result(math_gc_out, 'Group C DPO MATH')

    print('\nE7 完成')



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E4: 汇总所有结果
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

def load_acc(path):
    if not os.path.isfile(path):
        return None, 0
    d = json.load(open(path))
    acc = d.get('accuracy', d.get('macro_avg_accuracy'))
    total = d.get('total', 0)
    return acc, total

# 本次补充评测
supplement = [
    ('1.5B Baseline', 'logs/eval_supplement_1.5b_gsm8k.json', 'logs/eval_supplement_1.5b_math.json'),
    ('7B Baseline', 'logs/eval_supplement_7b_gsm8k.json', 'logs/eval_supplement_7b_math.json'),
    ('Group A DPO', 'logs/eval_supplement_group_a_dpo_gsm8k.json', 'logs/eval_supplement_group_a_dpo_math.json'),
    ('Group C DPO', 'logs/eval_supplement_group_c_dpo_gsm8k.json', 'logs/eval_supplement_group_c_dpo_math.json'),
    ('Targeted DPO', 'logs/eval_supplement_targeted_dpo_gsm8k.json', 'logs/eval_supplement_targeted_dpo_math.json'),
]

# 之前已有结果（logs 2/）
existing = [
    ('Group A SFT', 'logs 2/group_a_sft_gsm8k.json', 'logs 2/group_a_sft_math.json'),
    ('Group B SFT', 'logs 2/gsm8k_sft.json', 'logs 2/math_sft.json'),
    ('Group B DPO', 'logs 2/gsm8k_result.json', 'logs 2/math_result.json'),
    ('Group D', 'logs 2/group_d_gsm8k.json', 'logs 2/group_d_math.json'),
]

print('=' * 70)
print(f'{"模型":20s} {"GSM8K":>10s} {"MATH":>10s} {"n":>6s}')
print('-' * 70)

all_rows = []
for label, gsm_path, math_path in supplement + existing:
    gsm_acc, gsm_n = load_acc(gsm_path)
    math_acc, math_n = load_acc(math_path)
    gsm_s = f'{gsm_acc:.1%}' if gsm_acc is not None else '—'
    math_s = f'{math_acc:.1%}' if math_acc is not None else '—'
    n_s = str(gsm_n or math_n or '—')
    print(f'{label:20s} {gsm_s:>10s} {math_s:>10s} {n_s:>6s}')
    all_rows.append({'model': label, 'gsm8k': gsm_acc, 'math': math_acc, 'n': gsm_n or math_n})

print('=' * 70)

# 保存汇总
os.makedirs('results/ablation', exist_ok=True)
with open('results/ablation/supplement_summary.json', 'w') as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)
print(f'\n汇总已保存: results/ablation/supplement_summary.json')
print('\n全部完成！')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E1: 1.5B Baseline 评测（GSM8K + MATH-500，n=200）
# 模型：Qwen/Qwen2.5-1.5B-Instruct（未经任何微调）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E1: 1.5B Baseline 评测')
print('='*60)

# GSM8K
gsm_out = 'logs/eval_supplement_1.5b_gsm8k.json'
if not is_eval_complete(gsm_out, EVAL_N):
    run_eval([
        'python3', 'eval/gsm8k_eval.py',
        '--model_path', BASE_MODEL,
        '--max_samples', EVAL_N,
        '--output', gsm_out,
    ] + BIT, '1.5B GSM8K')
print_result(gsm_out, '1.5B GSM8K')

# MATH-500
math_out = 'logs/eval_supplement_1.5b_math.json'
if not is_eval_complete(math_out, EVAL_N):
    run_eval([
        'python3', 'eval/math_eval.py',
        '--model_path', BASE_MODEL,
        '--max_samples', EVAL_N,
        '--output', math_out,
    ] + BIT, '1.5B MATH')
print_result(math_out, '1.5B MATH')

print('\nE1 完成')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E2: 7B Baseline 评测（DashScope API，GSM8K + MATH-500，n=200）
# 模型：qwen2.5-7b-instruct（通过 API 调用）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

api_key = os.environ.get('DASHSCOPE_API_KEY', '')
if not api_key:
    print('⚠️ DASHSCOPE_API_KEY 未设置，跳过 E2（7B 评测）')
    print('  请在 Colab Secrets 中设置 DASHSCOPE_API_KEY')
else:
    print('\n' + '='*60)
    print('  E2: 7B Baseline 评测 (DashScope API)')
    print('='*60)

    DASHSCOPE_URL = 'https://dashscope.aliyuncs.com/compatible-mode/v1'

    # 7B GSM8K (API)
    gsm7b_out = 'logs/eval_supplement_7b_gsm8k.json'
    if not is_eval_complete(gsm7b_out, EVAL_N):
        run_eval([
            'python3', '-u', 'eval/gsm8k_api_eval.py',
            '--api_base_url', DASHSCOPE_URL,
            '--api_key', api_key,
            '--model', 'qwen2.5-7b-instruct',
            '--max_samples', EVAL_N,
            '--output', gsm7b_out,
        ], '7B GSM8K')
    print_result(gsm7b_out, '7B GSM8K')

    # 7B MATH-500 (API)
    math7b_out = 'logs/eval_supplement_7b_math.json'
    if not is_eval_complete(math7b_out, EVAL_N):
        run_eval([
            'python3', '-u', 'eval/math_api_eval.py',
            '--api_base_url', DASHSCOPE_URL,
            '--api_key', api_key,
            '--model', 'qwen2.5-7b-instruct',
            '--max_samples', EVAL_N,
            '--output', math7b_out,
        ], '7B MATH')
    print_result(math7b_out, '7B MATH')

    print('\nE2 完成')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E3: Group A DPO 评测（GSM8K + MATH-500，n=200）
# 模型：outputs/group_a/merged（LoRA + 单段SFT + Standard DPO）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# 确保 G_A_MERGED 变量存在且模型可用
if 'G_A_MERGED' not in dir() or not G_A_MERGED:
    G_A_MERGED = 'outputs/group_a/merged'

if not os.path.isfile(f'{G_A_MERGED}/config.json'):
    print('⚠️ Group A DPO 模型不存在，跳过 E3')
    print(f'  检查: {G_A_MERGED}/config.json')
else:
    print('\n' + '='*60)
    print(f'  E3: Group A DPO 评测 (模型: {G_A_MERGED})')
    print('='*60)

    # GSM8K
    gsm_ga_out = 'logs/eval_supplement_group_a_dpo_gsm8k.json'
    if not is_eval_complete(gsm_ga_out, EVAL_N):
        run_eval([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', G_A_MERGED,
            '--max_samples', EVAL_N,
            '--output', gsm_ga_out,
        ] + BIT, 'Group A DPO GSM8K')
    print_result(gsm_ga_out, 'Group A DPO GSM8K')

    # MATH-500
    math_ga_out = 'logs/eval_supplement_group_a_dpo_math.json'
    if not is_eval_complete(math_ga_out, EVAL_N):
        run_eval([
            'python3', 'eval/math_eval.py',
            '--model_path', G_A_MERGED,
            '--max_samples', EVAL_N,
            '--output', math_ga_out,
        ] + BIT, 'Group A DPO MATH')
    print_result(math_ga_out, 'Group A DPO MATH')

    print('\nE3 完成')
